# 🔐 Create Cardano Wallet — Step-by-Step Guide

Notebook này tạo một ví Cardano hoàn chỉnh theo workflow `create-wallet.md`:

| Bước | Mô tả |
|------|-------|
| 1 | Sinh mnemonic 24 từ |
| 2 | Derive payment signing key |
| 3 | Extract payment verification key |
| 4 | Derive stake signing key |
| 5 | Extract stake verification key |
| 6 | Build payment address |
| 7 | Build stake address |

> **⚠️ Cảnh báo bảo mật:** `mnemonic.txt` và `*.skey` là file bí mật — KHÔNG chia sẻ, lưu offline an toàn.

## 0. Cấu hình môi trường

In [ ]:
import subprocess
import os
import json
from pathlib import Path

# ── Đường dẫn cardano-cli.exe ──────────────────────────────
CARDANO_CLI = r"d:\Blockchain\tooldev\cardano-cli-win64\cardano-cli-11.0.0.0-win64\cardano-cli.exe"

# ── Thư mục lưu key files ──────────────────────────────────
WALLET_DIR = Path(r"d:\Blockchain\tooldev\cardano-cli-win64\cardano-cli-11.0.0.0-win64\wallet-keys")
WALLET_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(WALLET_DIR)

print(f"✅ cardano-cli : {CARDANO_CLI}")
print(f"✅ Wallet dir  : {WALLET_DIR}")
print(f"✅ CLI version :")
result = subprocess.run([CARDANO_CLI, "--version"], capture_output=True, text=True)
print(result.stdout.strip())

---
## Bước 1 — Sinh mnemonic seed phrase (24 từ)

Lệnh:
```bash
cardano-cli key generate-mnemonic --size 24 --out-file mnemonic.txt
```

Mnemonic là **gốc** của toàn bộ ví. Bất kỳ ai có mnemonic đều có thể khôi phục toàn bộ key → giữ tuyệt đối bí mật.

In [ ]:
mnemonic_file = WALLET_DIR / "mnemonic.txt"

result = subprocess.run(
    [CARDANO_CLI, "key", "generate-mnemonic",
     "--size", "24",
     "--out-file", str(mnemonic_file)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã sinh mnemonic 24 từ →", mnemonic_file.name)
    print()
    # Đọc và hiển thị (chỉ demo — xoá print sau khi dùng thật!)
    with open(mnemonic_file, "r") as f:
        mnemonic_data = json.load(f)
    print("📋 Mnemonic (DEMO — xoá sau khi lưu offline):")
    print(json.dumps(mnemonic_data, indent=2))

---
## Bước 2 — Derive payment signing key (`payment.skey`)

Lệnh:
```bash
cardano-cli key derive-from-mnemonic \
  --mnemonic-from-file mnemonic.txt \
  --payment-key-with-number 0 \
  --account-number 0 \
  --signing-key-file payment.skey
```

Payment signing key dùng để **ký giao dịch chi ADA**. `--payment-key-with-number 0` lấy key đầu tiên trong account 0.

In [ ]:
payment_skey = WALLET_DIR / "payment.skey"

result = subprocess.run(
    [CARDANO_CLI, "key", "derive-from-mnemonic",
     "--mnemonic-from-file", str(mnemonic_file),
     "--payment-key-with-number", "0",
     "--account-number", "0",
     "--signing-key-file", str(payment_skey)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã tạo payment.skey →", payment_skey.name)
    # Hiển thị cấu trúc (type, description — KHÔNG hiển thị cborHex)
    with open(payment_skey, "r") as f:
        data = json.load(f)
    print(f"   type        : {data.get('type')}")
    print(f"   description : {data.get('description')}")
    print(f"   cborHex     : [HIDDEN — bí mật]")

---
## Bước 3 — Extract payment verification key (`payment.vkey`)

Lệnh:
```bash
cardano-cli key verification-key \
  --signing-key-file payment.skey \
  --verification-key-file payment.vkey
```

Verification key là **public key** — an toàn để chia sẻ. Dùng để build address và verify chữ ký.

In [ ]:
payment_vkey = WALLET_DIR / "payment.vkey"

result = subprocess.run(
    [CARDANO_CLI, "key", "verification-key",
     "--signing-key-file", str(payment_skey),
     "--verification-key-file", str(payment_vkey)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã tạo payment.vkey →", payment_vkey.name)
    with open(payment_vkey, "r") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2))

---
## Bước 4 — Derive stake signing key (`stake.skey`)

Lệnh:
```bash
cardano-cli key derive-from-mnemonic \
  --mnemonic-from-file mnemonic.txt \
  --stake-key-with-number 0 \
  --account-number 0 \
  --signing-key-file stake.skey
```

Stake signing key dùng để **ký các thao tác staking** (delegate, register stake, withdraw rewards).

In [ ]:
stake_skey = WALLET_DIR / "stake.skey"

result = subprocess.run(
    [CARDANO_CLI, "key", "derive-from-mnemonic",
     "--mnemonic-from-file", str(mnemonic_file),
     "--stake-key-with-number", "0",
     "--account-number", "0",
     "--signing-key-file", str(stake_skey)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã tạo stake.skey →", stake_skey.name)
    with open(stake_skey, "r") as f:
        data = json.load(f)
    print(f"   type        : {data.get('type')}")
    print(f"   description : {data.get('description')}")
    print(f"   cborHex     : [HIDDEN — bí mật]")

---
## Bước 5 — Extract stake verification key (`stake.vkey`)

Lệnh:
```bash
cardano-cli key verification-key \
  --signing-key-file stake.skey \
  --verification-key-file stake.vkey
```

Stake verification key (public) — dùng để build stake address và payment address có delegation.

In [ ]:
stake_vkey = WALLET_DIR / "stake.vkey"

result = subprocess.run(
    [CARDANO_CLI, "key", "verification-key",
     "--signing-key-file", str(stake_skey),
     "--verification-key-file", str(stake_vkey)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã tạo stake.vkey →", stake_vkey.name)
    with open(stake_vkey, "r") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2))

---
## Bước 6 — Build payment address (`payment.addr`)

Lệnh:
```bash
cardano-cli address build \
  --payment-verification-key-file payment.vkey \
  --stake-verification-key-file stake.vkey \
  --mainnet \
  --out-file payment.addr
```

Payment address là địa chỉ **nhận ADA** — kết hợp payment key + stake key để tự động delegate reward. An toàn chia sẻ công khai.

> Đổi `--mainnet` thành `--testnet-magic <magic>` nếu dùng testnet.

In [ ]:
payment_addr = WALLET_DIR / "payment.addr"

result = subprocess.run(
    [CARDANO_CLI, "address", "build",
     "--payment-verification-key-file", str(payment_vkey),
     "--stake-verification-key-file", str(stake_vkey),
     "--mainnet",
     "--out-file", str(payment_addr)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    addr = payment_addr.read_text().strip()
    print("✅ Đã tạo payment.addr →", payment_addr.name)
    print()
    print(f"💰 Payment Address:")
    print(f"   {addr}")
    print()
    # Phân tích prefix
    if addr.startswith("addr1q"):
        print("   → Mainnet, Shelley-era (bech32)")
    elif addr.startswith("addr_test1"):
        print("   → Testnet address")
    else:
        print(f"   → Prefix: {addr[:10]}...")

---
## Bước 7 — Build stake address (`stake.addr`)

Lệnh:
```bash
cardano-cli conway stake-address build \
  --stake-verification-key-file stake.vkey \
  --mainnet \
  --out-file stake.addr
```

Stake address dùng cho các thao tác **staking** — register stake, delegate, withdraw reward. Không nhận ADA trực tiếp.

In [ ]:
stake_addr = WALLET_DIR / "stake.addr"

result = subprocess.run(
    [CARDANO_CLI, "conway", "stake-address", "build",
     "--stake-verification-key-file", str(stake_vkey),
     "--mainnet",
     "--out-file", str(stake_addr)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    addr = stake_addr.read_text().strip()
    print("✅ Đã tạo stake.addr →", stake_addr.name)
    print()
    print(f"🏦 Stake Address:")
    print(f"   {addr}")
    print()
    if addr.startswith("stake1"):
        print("   → Mainnet stake address (bech32)")
    elif addr.startswith("stake_test1"):
        print("   → Testnet stake address")

---
## 📋 Tổng kết — Kiểm tra tất cả file đã tạo

In [ ]:
expected_files = [
    ("mnemonic.txt",   "🔒 BÍ MẬT — 24-word seed phrase"),
    ("payment.skey",   "🔒 BÍ MẬT — Payment signing key"),
    ("payment.vkey",   "🌐 Public  — Payment verification key"),
    ("stake.skey",     "🔒 BÍ MẬT — Stake signing key"),
    ("stake.vkey",     "🌐 Public  — Stake verification key"),
    ("payment.addr",   "💰 Public  — Payment address (nhận ADA)"),
    ("stake.addr",     "🏦 Public  — Stake address (staking)"),
]

print("=" * 70)
print("  📦 WALLET FILES — Tổng kết")
print("=" * 70)
print()

all_ok = True
for filename, desc in expected_files:
    filepath = WALLET_DIR / filename
    if filepath.exists():
        size = filepath.stat().st_size
        print(f"  ✅ {filename:<18} {size:>6} bytes  {desc}")
    else:
        print(f"  ❌ {filename:<18} {'MISSING':>8}      {desc}")
        all_ok = False

print()
if all_ok:
    print("  🎉 Ví Cardano đã được tạo thành công!")
else:
    print("  ⚠️  Một số file bị thiếu — kiểm tra lỗi ở các bước trên.")

print()
print("=" * 70)
print("  🔐 GHI NHỚ BẢO MẬT")
print("=" * 70)
print("  • mnemonic.txt, *.skey → KHÔNG chia sẻ, lưu offline (paper/metal)")
print("  • payment.addr, *.vkey → An toàn chia sẻ công khai")
print("  • Backup mnemonic ở nhiều nơi an toàn khác nhau")
print("=" * 70)

---
## 🔍 Bonus — Hiển thị địa chỉ ví (copy-paste)

In [ ]:
payment_addr_str = (WALLET_DIR / "payment.addr").read_text().strip()
stake_addr_str   = (WALLET_DIR / "stake.addr").read_text().strip()

print("━" * 60)
print("  💰 PAYMENT ADDRESS (chia sẻ để nhận ADA)")
print("━" * 60)
print()
print(f"  {payment_addr_str}")
print()
print("━" * 60)
print("  🏦 STAKE ADDRESS (dùng cho staking)")
print("━" * 60)
print()
print(f"  {stake_addr_str}")
print()
print("━" * 60)

---
## 🧹 Cleanup — Xóa toàn bộ file đã tạo

> **⚠️ Chỉ chạy sau khi đã backup mnemonic ra giấy an toàn!** Cell này xóa sạch mọi file trong `wallet-keys/`."

In [ ]:
CONFIRM = False  # ← Đổi True để xóa thật

files_to_delete = [
    "mnemonic.txt",
    "payment.skey", "payment.vkey",
    "stake.skey", "stake.vkey",
    "payment.addr", "stake.addr",
]

if CONFIRM:
    import gc

    print("🧹 Bước 1: Xóa file trên disk...\n")
    for fname in files_to_delete:
        fpath = WALLET_DIR / fname
        if fpath.exists():
            fpath.unlink()
            print(f"  ❌ Deleted: {fname}")
        else:
            print(f"  ➖ Skip (not found): {fname}")

    print("\n🧹 Bước 2: Xóa biến trong RAM...\n")
    # Xóa mọi biến chứa key/mnemonic/addr
    for var in [
        "mnemonic_file", "mnemonic_data",
        "payment_skey", "payment_vkey",
        "stake_skey", "stake_vkey",
        "payment_addr", "stake_addr",
        "payment_addr_str", "stake_addr_str",
        "addr",
    ]:
        if var in globals():
            del globals()[var]
            print(f"  ❌ Del var: {var}")

    # Force garbage collector dọn RAM
    collected = gc.collect()
    print(f"\n  🗑️  gc.collect() reclaimed {collected} objects")

    print("\n🧹 Bước 3: Xóa lịch sử output...\n")
    # Clear In/Out history khỏi RAM
    from IPython import get_ipython
    ip = get_ipython()
    if ip:
        ip.history_manager.reset()
        print("  ❌ Cleared input/output history")

    print("\n✅ Đã xóa sạch file + RAM. Mnemonic đã có trên giấy — giữ an toàn!")
    print("💡 Để dọn sạch hoàn toàn: Kernel → Restart (Ctrl+Shift+.)")
else:
    print("⏸️  Cleanup đang tắt (CONFIRM=False)")
    print("   Đổi CONFIRM=True để xóa. Chỉ chạy sau khi đã viết mnemonic ra giấy!")

---
# 📝 Build & Sign Transaction — Giao dịch Cardano

Phần này thực hiện workflow `build-and-sign-tx.md`: build giao dịch, tính fee, ký, lấy txID và submit.

| Bước | Mô tả |
|------|-------|
| 8 | Cấu hình tham số giao dịch |
| 9A | Build tx với auto-fee (cần node/protocol-params) |
| 9B | Build raw tx offline + tính fee thủ công |
| 10 | Ký giao dịch (`tx.signed`) |
| 11 | Lấy Transaction ID |
| 12 | Submit giao dịch (cần node hoặc external service) |
| 13 | Multi-signature — partial signing (witness + assemble) |
| 14 | Helper: ADA ↔ Lovelace |

> **Đơn vị:** 1 ADA = 1,000,000 lovelace